# 13.16 獎勵提高，如何知道任務真的完成？


任務是「只回數字：1+2=?」。短答「3」合格，長句「我已經仔細思考過，答案是4」既算錯也違反格式。若獎勵只按字元數，長句卻得高分。策略利用評分弱點拿分，卻沒完成真正任務，叫**獎勵鑽漏洞（reward hacking）**；不需要假設它有人的欺騙意圖。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
answers = ["3", "我已經仔細思考過，答案是4。"]
scores = [len(answer) for answer in answers]
chosen = max(range(len(answers)), key=lambda index: scores[index])
print("評分規則挑中", answers[chosen])
print("符合只回數字且算對", answers[chosen] == str(1 + 2))

`len`數字元，`max`按scores選最高分候選，因此挑中長句，任務檢查為`False`。這是人工設計的評分漏洞，不是本章已訓練policy的實測。固定reference或PPO裁切，都不能替錯誤獎勵改判準。

實測要用不依賴reward分數的任務檢查。本章選卡的最後18題，每種條件6題，成功是選中符合完整要求的預寫卡：

| 條件 | 示範起點 | PPO分支 | DPO分支 |
| --- | ---: | ---: | ---: |
| 只回數字 | 6／6 | 6／6 | 6／6 |
| 用一句話說明 | 0／6 | 0／6 | 0／6 |
| 缺購買數量，應求補資訊 | 0／6 | 6／6 | 6／6 |
| 完整要求 | 6／18 | 12／18 | 12／18 |

兩條分支新增了求補資訊的選擇，仍未學好句子條件。「一句話」在這裡只要求帶算式與結果，不衡量解釋深度。評分員把最後90個偏好對全部排對，策略卻未全過；評得對與選得對仍是兩件事。

這輪沒有觀察到policy鑽已學評分員的漏洞；表中的問題是說明條件尚未學成。真正檢查時保留原回答，另核對真值、範圍、格式與必要資訊，而非只讓同一評分員為自己背書。

練習只把 `scores` 列表推導式中的 `len(answer)` 換成 `int(answer == str(1 + 2))`，保留 `for answer in answers`，核對改選3。字串相等適合這個有限數字題，不能直接用於多種合理說法的開放解釋。

<details>
<summary>補充：來源、完整設定與既有實測紀錄</summary>

這份任務的「一句話說明」只檢查候選是否包含算式與結果的句子，不衡量解釋加法原理的深度。即使在這麼有限的要求上，兩支仍失敗。看`1+2`的說明題：評分員將句子卡排最高，原始分數約13.5565，短答卡約8.7679；PPO卻給短答約0.7928、句子約0.2014，取最高機率時仍選`3`。答對數字與完成要求，確實是兩個結果。

評分員在最後90個偏好對全部排對，但不能把這90／90當成回答策略的90／90。這次說明題在訓練側也0／44通過，不只是新家族泛化失敗；固定短訓配方尚未學好這個條件。它示範的是「評分員排序正確，選擇模型仍未達標」，沒有觀察到策略利用已學評分員的漏洞。上面的長度反例與正式實驗，必須分開報告。

</details>
